References:



https://colab.research.google.com/github/dvgodoy/FineTuningLLMs/blob/main/AppendixB.ipynb#scrollTo=eL5kArWv1KEi  

## Floating-Point Numbers: From Bits to Value

> Covers normal values, zero, subnormals, infinity, and `NaN`. Rounding comes later.

---

### 1. Binary Basics

- Place values: `... 8  4  2  1 . 1/2  1/4  1/8 ...`
- `1` = include that place value; `0` = skip it.
- **Normalized binary:** exactly one `1` before the binary point.
- To keep a value unchanged while rewriting it:
  - move the point left `n` places → multiply by `2^n`
  - move the point right `n` places → multiply by `2^-n`

```text
11.1₂  = 1.11₂ × 2¹    (point moved left 1)
0.011₂ = 1.1₂  × 2⁻²   (point moved right 2)
```

---

### 2. Same Bits, Different Type

```text
10000011₂ as uint8 =  128 + 2 + 1 = 131    (range 0 to 255)
10000011₂ as int8  = −128 + 2 + 1 = −125   (range −128 to 127)
```

- `int8` uses two's complement: its leftmost bit is worth `−128`, not `+128`.
- FP32's exponent field is read as an **unsigned** code, like `uint8`.

<img src="https://raw.githubusercontent.com/dvgodoy/FineTuningLLMs/main/images/appendixB/int8_example.png" width="300">

---

### 3. The Float Idea

A float is scientific notation in base 2:

```text
sign × significand × 2^e
```

For a normal value:

$$
\text{value} = (-1)^s \times (1 + f) \times 2^e
$$

| Symbol | Meaning |
|---|---|
| `s` | sign bit: `0` positive, `1` negative |
| `e` | actual exponent: moves the binary point |
| `F` | stored fraction bits: `f₁f₂f₃...` |
| `f` | value of those bits: `f₁×2⁻¹ + f₂×2⁻² + f₃×2⁻³ + ...` |
| `1.F` | the **significand** `= 1 + f` (older name: “mantissa”) |

Example:

```text
F = 110...
f = 1/2 + 1/4 = 0.75
significand = 1.11₂ = 1.75
```

---

### 4. FP32 Layout

```text
[ sign: 1 bit ] [ exponent code E: 8 bits ] [ fraction bits F: 23 bits ]
```

The fields are separate. No binary point is stored; the exponent rebuilds the value.

For a **normal** FP32 value (`E = 1` to `254`):

$$
\text{value} = (-1)^s \times (1.F)_2 \times 2^{E - 127}
$$

#### Exponent

- `E` = stored 8-bit unsigned exponent **code**
- `e` = actual exponent used in `× 2^e`

```text
Encode: e → add 127 → stored E → write E as 8 bits
Decode: E bits → E as unsigned integer → subtract 127 → e
```

```text
Store: E = e + 127
Read:  e = E − 127
```

**Why add 127?** Actual exponents can be negative, but the stored 8-bit unsigned code can only be `0` to `255`. The bias maps normal exponents `−126` to `127` into codes `1` to `254`. Codes `0` and `255` are reserved for special values.

```text
e = −2  →  E = 125
e =  0  →  E = 127
e =  1  →  E = 128
```

> **Watch out:** Subtract `127` only from the stored code `E`.  
> In `1.11₂ × 2¹`, the `1` is already the actual exponent `e`.

#### Fraction

- FP32 stores the 23 fraction bits `F`.
- For normal values, the leading `1.` is **hidden** because it is always present.
- Rebuild the significand as `1.F`.

```text
23 stored fraction bits + 1 hidden leading bit
= 24 significant binary bits
```

<img src="https://raw.githubusercontent.com/dvgodoy/FineTuningLLMs/main/images/appendixB/bf16_example.png" width="428">

*In this BF16 example: `E = 131 → e = 4`; `f = 0.4609375`; so `1.4609375 × 2⁴ = 23.375`. The image calls the fraction bits “Mantissa (M)”.*

---

### 5. Full Loop: Encode and Decode 3.5

`3.5` is exactly representable, so no rounding is needed.

#### Encode

```text
3.5 → 11.1₂                 (3 = 11₂, 0.5 = 0.1₂)
    → 1.11₂ × 2¹            (normalize)

sign bit:            s = 0
actual exponent:     e = 1
stored exponent:     E = e + 127 = 128 = 10000000₂
stored fraction:     F = 11, then pad with 0s to 23 bits

FP32 bits:
0 | 10000000 | 110 0000 0000 0000 0000 0000

hex: 0x40600000
```

#### Decode

```text
Input bits:
0 | 10000000 | 110 0000 0000 0000 0000 0000

sign:     0 → positive
E bits:   10000000₂ → E = 128 → e = 128 − 127 = 1
F bits:   110... → significand = 1.110...₂ = 1.75

value = +1.75 × 2¹ = 3.5
```

```text
Encode: number → normalized binary → FP32 bits
Decode: FP32 bits → normalized binary → number
```

---

### 6. Special FP32 Values

| Exponent code `E` | Fraction `F` | Meaning |
|---|---|---|
| `0` | all `0`s | `+0` or `−0` |
| `0` | not all `0`s | subnormal number |
| `1` to `254` | anything | normal number |
| `255` | all `0`s | `+∞` or `−∞` |
| `255` | not all `0`s | `NaN` (“not a number”) |

#### Subnormals: values below the smallest normal number

When `E = 0`, **do not** calculate `e = 0 − 127`.

Instead:

- there is **no hidden leading `1`**
- the significand is `0.F`
- FP32 uses the fixed scale `2⁻¹²⁶`

$$
\text{subnormal value} = (-1)^s \times (0.F)_2 \times 2^{-126}
$$

```text
smallest normal:
E = 1, F = all 0s  →  1.000...₂ × 2⁻¹²⁶

smallest positive subnormal:
E = 0, F = 000...001  →  2⁻²³ × 2⁻¹²⁶ = 2⁻¹⁴⁹
```

The fraction bits count tiny units:

```text
F ends in 001  →  1 × 2⁻¹⁴⁹
F ends in 010  →  2 × 2⁻¹⁴⁹
F ends in 011  →  3 × 2⁻¹⁴⁹
                = 1 × 2⁻¹⁴⁸ + 1 × 2⁻¹⁴⁹
```

So values shrink smoothly toward zero in steps of `2⁻¹⁴⁹`. This is **gradual underflow**.

```text
E = 255, F = all 0s      → ±infinity (usual result of overflow)
E = 255, F not all 0s    → NaN, e.g. 0 ÷ 0
```

---

### 7. Range vs. Precision

- **Exponent bits → range:** how small or large magnitudes can be.
- **Fraction bits → precision:** spacing between nearby values.

Normal FP32 limits:

```text
largest normal:
E = 254, F = all 1s
(2 − 2⁻²³) × 2¹²⁷ ≈ 3.4 × 10³⁸

smallest normal:
E = 1, F = all 0s
1.0 × 2⁻¹²⁶ ≈ 1.2 × 10⁻³⁸
```

Spacing doubles whenever `e` increases by `1`:

```text
[1, 2)        e = 0    spacing = 2⁻²³ ≈ 1.2 × 10⁻⁷
[2, 4)        e = 1    spacing = 2⁻²²
[2²⁴, 2²⁵)    e = 24   spacing = 2
```

At and above `16,777,216 = 2²⁴`, odd integers cannot be stored in FP32.

**Precision is relative:** FP32 gives roughly **7 decimal significant digits** for normal values.

---

### 8. FP32 vs. BF16 vs. FP16

<img src="https://raw.githubusercontent.com/dvgodoy/FineTuningLLMs/main/images/appendixB/types_comparison.png" width="600">

All use the same normal-value idea:

$$
\text{value} = (-1)^s \times (1.F)_2 \times 2^{E-\text{bias}}
$$

$$
\text{bias} = 2^{(\text{exponent bits}-1)} - 1
$$

| Type | Exponent bits | Fraction bits | Significant bits* | Bias | Largest finite value | In short |
|---|---:|---:|---:|---:|---:|---|
| FP32 | 8 | 23 | 24 | 127 | ≈ `3.4 × 10³⁸` | wide range, high precision |
| BF16 | 8 | 7 | 8 | 127 | ≈ `3.4 × 10³⁸` | FP32’s normal exponent range, far less precision |
| FP16 | 5 | 10 | 11 | 15 | `65,504` | smaller range, more precision than BF16 |

\*For normal values, includes the hidden leading `1`.

---

### 9. Connection to Quantization

- **BF16 vs. FP16:** BF16 has FP32’s normal exponent range and largest finite magnitude, so it is much less likely to overflow than FP16. FP16 overflows above `65,504`.
- Both BF16 and FP16 use **2 bytes per parameter**: 7B parameters need about **14 GB** for weights alone.
- Floats have an exponent, so their spacing grows with magnitude (**relative precision**). INT8/INT4 have no exponent, so their integer values are evenly spaced.
- Quantization uses a **scale `S`** to stretch that fixed integer range to cover the real weight values.